# LLM 10일차 (1) — 체크포인팅 : `AsyncSqliteSaver` · `thread_id` 로 대화 기억하기

교재 `AI Agent` **p.130~137** (체크포인팅 — 용어 · 쓰임 · `AsyncSqliteSaver` · 핵심 변화 포인트 3가지) 와 수업 필기 `LangGraph_실습6`(체크포인터를 붙인 완성본)을 정리한 노트입니다.

```python
async with AsyncSqliteSaver.from_conn_string('multi_agent.sqlite') as memory:   # DB 연결 (블록이 끝나면 자동으로 닫힘)
    agent = graph.compile(checkpointer=memory)                                   # 외부 기억 장치 장착
    config = {'configurable': {'thread_id': 'user-8888'}}                         # 대화방 번호
    final_state = await agent.ainvoke({'messages': [HumanMessage(질문)]}, config=config)   # 새 질문 하나만
```

- MySQL 판(`AIOMySQLSaver`, 필기 `실습7`)은 [`langgraph_checkpoint_mysql.ipynb`](./langgraph_checkpoint_mysql.ipynb) · 조건부 분기 + 비동기 + 체크포인팅을 합친 판(필기 `실습8`)은 [`langgraph_rag_router_async_checkpoint.ipynb`](./langgraph_rag_router_async_checkpoint.ipynb)
- 원본에서 고친 부분 · 보충은 **📝** 로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다.
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩** · **가짜 LLM** 으로 실제로 돌린 셀 — 답변 문장은 gpt-5-nano 와 다르다. **체크포인트 저장 · 복원 · thread_id 분리 · 오류는 진짜와 같다** |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langgraph-checkpoint-sqlite` | `AsyncSqliteSaver` | `pip install langgraph-checkpoint-sqlite` |
| `aiosqlite` | 비동기 SQLite 연결 (위 패키지와 함께 설치됨) | `pip install aiosqlite` |
| `langgraph` · `langchain-openai` · `langchain-community` · `langchain-chroma` | 그래프 · LLM · CSV · 벡터 저장소 (Day 053~054) | |
| `langchain-huggingface` | (🧪 키 없이) 로컬 임베딩 | `pip install langchain-huggingface sentence-transformers` |

1. 같은 폴더에 `faq_manual.csv` · `.env`(`OPENAI_API_KEY`) — **`.env` 는 올리지 않는다**
2. 맨 위부터 실행. 실행하면 같은 폴더에 `*.sqlite` 파일(체크포인트 DB)이 생긴다 — 대화 기록이 들어 있으니 저장소에 올리지 말 것
3. 정리 환경 : `langgraph 1.2.21` · `langgraph-checkpoint 4.3.1` · `langgraph-checkpoint-sqlite 3.1.3` · `aiosqlite 0.22.1`

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph-checkpoint-sqlite aiosqlite

In [2]:
import os, asyncio, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from typing import Annotated
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langgraph.graph import StateGraph, END, add_messages
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

loaded = load_dotenv(find_dotenv(usecwd=True), override=True)          # 키 값은 출력하지 않는다
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False


## 1. 용어 · 쓰임 (교재 p.130~132)

| 용어 | 뜻 | 비유 |
|---|---|---|
| 상태 (State) | 워크플로우의 모든 정보를 담는 공유 메모리 | 지금 게임 화면 |
| 체크포인트 (Checkpoint) | 한 단계가 끝난 직후의 상태 **그 자체** (결과물) | 저장 파일 |
| 체크포인팅 (Checkpointing) | 상태를 체크포인트로 저장하고 다시 불러와 이어 가는 **과정** (행위) | 저장하기 / 불러오기 |
| 체크포인터 (Checkpointer) | 어디에 · 어떻게 저장할지 아는 **도구** — `InMemorySaver` · `AsyncSqliteSaver` · `AIOMySQLSaver` … | 세이브 슬롯이 있는 메모리 카드 |
| `thread_id` | 체크포인트를 묶는 **대화방 번호** | 세이브 슬롯 번호 |

쓰임 : ① 대화 기억(멀티턴) ② 긴 작업이 중간에 멈춰도 이어 가기 ③ 오류가 난 단계부터 다시 시도

## 2. 작은 그래프로 확인 — 무엇이 언제 저장되나

LLM 없이 노드 2개(`a` : `context` 를 채움 → `b` : 답 메시지)짜리 그래프로 확인했다.

In [3]:
class DemoState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []   # 리듀서 있음 → 누적
    context: str = ''                                           # 리듀서 없음 → 덮어쓰기

async def node_a(state: DemoState) -> dict:
    return {'context': f"'{state.messages[-1].content}' 의 검색 결과"}

async def node_b(state: DemoState) -> dict:
    return {'messages': [AIMessage(f'답변 (받은 messages {len(state.messages)}개)')]}

demo = StateGraph(DemoState)
demo.add_node('a', node_a); demo.add_node('b', node_b)
demo.set_entry_point('a'); demo.add_edge('a', 'b'); demo.add_edge('b', END)

DB = 'demo_checkpoint.sqlite'
if os.path.exists(DB):
    os.remove(DB)                          # 노트를 다시 실행해도 같은 결과가 나오도록 새 DB 로 시작

In [4]:
async with AsyncSqliteSaver.from_conn_string(DB) as memory:
    app = demo.compile(checkpointer=memory)
    config = {'configurable': {'thread_id': 'user-8888'}}

    for q in ['환불하고 싶어요', '그거 전액 받으려면?']:
        result = await app.ainvoke({'messages': [HumanMessage(q)]}, config=config)   # 새 질문 하나만 넣는다
        print(f'질문 : {q}')
        print('  messages :', [f'{type(m).__name__[:-7]}: {m.content}' for m in result['messages']])

    # 저장된 '지금 상태' 와 '저장 파일 목록'
    snapshot = await app.aget_state(config)
    print('\n현재 체크포인트 상태 키 :', list(snapshot.values), '| 다음 실행할 노드 :', snapshot.next)
    history = [h async for h in app.aget_state_history(config)]
    print(f'체크포인트 수 : {len(history)}개 (최신 → 처음)')
    for h in history:
        print(f"  step {h.metadata['step']:>2}  source={h.metadata['source']:<5}  messages {len(h.values.get('messages', []))}개  다음={h.next}")

질문 : 환불하고 싶어요
  messages : ['Human: 환불하고 싶어요', 'AI: 답변 (받은 messages 1개)']
질문 : 그거 전액 받으려면?
  messages : ['Human: 환불하고 싶어요', 'AI: 답변 (받은 messages 1개)', 'Human: 그거 전액 받으려면?', 'AI: 답변 (받은 messages 3개)']

현재 체크포인트 상태 키 : ['messages', 'context'] | 다음 실행할 노드 : ()
체크포인트 수 : 8개 (최신 → 처음)
  step  6  source=loop   messages 4개  다음=()
  step  5  source=loop   messages 3개  다음=('b',)
  step  4  source=loop   messages 3개  다음=('a',)
  step  3  source=input  messages 2개  다음=('__start__',)
  step  2  source=loop   messages 2개  다음=()
  step  1  source=loop   messages 1개  다음=('b',)
  step  0  source=loop   messages 1개  다음=('a',)
  step -1  source=input  messages 0개  다음=('__start__',)


- 두 번째 질문에 **새 질문 하나만** 넣었는데 노드 `b` 가 받은 messages 는 3개였다 — 체크포인터가 `thread_id='user-8888'` 의 마지막 체크포인트를 불러와 새 질문과 합쳤다(교재 p.135 'thread_id → DB 조회 → 과거 상태 복원 → 새 질문과 병합')
- 체크포인트는 **한 턴에 4개** : 입력을 받은 순간(`source=input`) · 시작(`step 0`) · 노드 `a` 직후 · 노드 `b` 직후. 교재 p.131 '노드가 실행된 직후의 상태' 에 **입력 시점**이 더해진다
- 📝 '병합' 은 `add_messages` **리듀서가 있는 항목만** 덧붙이기다. 리듀서가 없는 `context` 같은 항목은 앞 턴 값이 남아 있다가 노드가 새로 쓸 때만 바뀐다 → §4

## 3. thread_id · config · `async with` 확인

In [5]:
async with AsyncSqliteSaver.from_conn_string(DB) as memory:
    app = demo.compile(checkpointer=memory)

    # (가) 다른 대화방은 기록이 섞이지 않는다
    other = await app.ainvoke({'messages': [HumanMessage('안녕하세요')]}, config={'configurable': {'thread_id': 'user-9999'}})
    print('(가) user-9999 messages :', len(other['messages']), '개')

    # (나) 체크포인터를 붙였는데 config(thread_id) 없이 부르면
    try:
        await app.ainvoke({'messages': [HumanMessage('안녕')]})
    except ValueError as e:
        print('(나) ValueError :', str(e).splitlines()[0])

# (다) async with 블록이 끝난 뒤에 같은 app 을 쓰면
try:
    await app.ainvoke({'messages': [HumanMessage('또 물어봐요')]}, config={'configurable': {'thread_id': 'user-8888'}})
except Exception as e:
    print('(다)', type(e).__name__, ':', e)

(가) user-9999 messages : 2 개
(나) ValueError : Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id
(다) ValueError : no active connection


In [6]:
# (라) 프로그램을 껐다 켠 것처럼 — 새로 연결해도 파일에 남은 기록이 이어진다
async with AsyncSqliteSaver.from_conn_string(DB) as memory:
    app = demo.compile(checkpointer=memory)
    result = await app.ainvoke({'messages': [HumanMessage('세 번째 질문')]}, config={'configurable': {'thread_id': 'user-8888'}})
    print('(라) user-8888 messages :', len(result['messages']), '개 →', result['messages'][-1].content)

import sqlite3
with sqlite3.connect(DB) as con:
    print('DB 테이블 :', [r[0] for r in con.execute("select name from sqlite_master where type='table'")])
    print('대화방별 체크포인트 수 :', list(con.execute('select thread_id, count(*) from checkpoints group by thread_id')))

(라) user-8888 messages : 6 개 → 답변 (받은 messages 5개)
DB 테이블 : ['checkpoints', 'writes']
대화방별 체크포인트 수 : [('user-8888', 12), ('user-9999', 4)]


| 확인 | 결과 |
|---|---|
| (가) 다른 `thread_id` | 기록이 섞이지 않는다 (2개) |
| (나) config 없이 `ainvoke` | `ValueError: Checkpointer requires … thread_id …` |
| (다) `async with` 블록 밖에서 사용 | `ValueError: no active connection` — 교재 p.134 ② '블록이 끝나면 연결이 자동으로 닫힌다' 그대로. 대화 루프 전체가 블록 **안**에 있어야 한다(필기 `main()` 이 그렇게 했다) |
| (라) 다시 연결 | 파일에 남은 기록에 이어서 6개 — 프로그램을 다시 켜도 기억한다 |
| DB 구조 | `checkpoints`(상태) · `writes`(노드별 쓰기) 두 테이블 |

## 4. 📝 리듀서 없는 항목은 '앞 턴 값' 이 남는다

In [7]:
async def skip_a(state: DemoState) -> dict:
    return {}                                  # 이번 턴에는 context 를 새로 쓰지 않는 노드 (예 : 검색이 필요 없는 질문)

skip = StateGraph(DemoState)
skip.add_node('a', skip_a); skip.add_node('b', node_b)
skip.set_entry_point('a'); skip.add_edge('a', 'b'); skip.add_edge('b', END)

async with AsyncSqliteSaver.from_conn_string(DB) as memory:
    app1 = demo.compile(checkpointer=memory)
    app2 = skip.compile(checkpointer=memory)          # 같은 DB · 같은 대화방
    cfg = {'configurable': {'thread_id': 'user-stale'}}
    r1 = await app1.ainvoke({'messages': [HumanMessage('환불 규정 알려줘')]}, cfg)
    r2 = await app2.ainvoke({'messages': [HumanMessage('고마워요')]}, cfg)
    print('1턴 context :', r1['context'])
    print('2턴 context :', r2['context'], '← 2턴에서는 아무도 쓰지 않았는데 남아 있다')

1턴 context : '환불 규정 알려줘' 의 검색 결과
2턴 context : '환불 규정 알려줘' 의 검색 결과 ← 2턴에서는 아무도 쓰지 않았는데 남아 있다


- 체크포인터는 상태 **전체**를 저장 · 복원한다. `messages` 처럼 리듀서가 있으면 덧붙고, `context` · `rephrased_query` · `needs_rag` 처럼 리듀서가 없으면 **앞 턴 값이 그대로** 다음 턴에 들어온다
- 그래서 '검색을 건너뛰는' 길이 있는 그래프(필기 `실습8`)라면 답변 노드가 `context` 를 믿기 전에 이번 턴에 채워진 값인지 확인해야 한다. 가장 쉬운 방법은 **분류 노드가 매 턴 `context: ''` 로 비우는 것**(`실습8` 정리 노트 §4)

## 5. 오류가 난 단계부터 이어 가기 (교재 p.132 ②)

In [8]:
fail_once = {'on': True}

async def flaky_b(state: DemoState) -> dict:
    if fail_once['on']:
        fail_once['on'] = False
        raise RuntimeError('LLM 서버 응답 없음 (일부러 낸 오류)')
    return {'messages': [AIMessage('다시 시도해서 성공한 답변')]}

flaky = StateGraph(DemoState)
flaky.add_node('a', node_a); flaky.add_node('b', flaky_b)
flaky.set_entry_point('a'); flaky.add_edge('a', 'b'); flaky.add_edge('b', END)

async with AsyncSqliteSaver.from_conn_string(DB) as memory:
    app = flaky.compile(checkpointer=memory)
    cfg = {'configurable': {'thread_id': 'user-retry'}}
    try:
        await app.ainvoke({'messages': [HumanMessage('환불 규정')]}, cfg)
    except RuntimeError as e:
        print('1차 실행 :', e)
    snap = await app.aget_state(cfg)
    print('멈춘 곳 — 다음 실행할 노드 :', snap.next, '| a 가 남긴 context :', snap.values['context'])
    result = await app.ainvoke(None, cfg)                    # 입력 None = '마지막 체크포인트부터 이어서'
    print('이어서 실행 :', [m.content for m in result['messages']])

1차 실행 : LLM 서버 응답 없음 (일부러 낸 오류)
멈춘 곳 — 다음 실행할 노드 : ('b',) | a 가 남긴 context : '환불 규정' 의 검색 결과
이어서 실행 : ['환불 규정', '다시 시도해서 성공한 답변']


- 노드 `b` 에서 오류가 나도 `a` 직후의 체크포인트가 남아 있어 `snap.next == ('b',)` 다. **`ainvoke(None, config)`** 로 부르면 `a` 를 다시 하지 않고 `b` 부터 이어 간다(LLM 호출을 아낀다)
- 📝 교재 p.132 '자동으로 되돌아가 다시 시도' 는 체크포인터가 **알아서 재시도하는 것이 아니다**. 이어 가는 호출(`ainvoke(None, …)`)을 코드에서 해 줘야 한다

## 6. 핵심 변화 포인트 3가지를 멀티 에이전트 RAG 챗봇에 (교재 p.135~137 · 필기 `실습6`)

| 포인트 | 기존 (`실습6` 비동기판, Day 054) | 체크포인팅 (`실습6` 완성본) |
|---|---|---|
| 1 메인 루프 | `inputs['messages'].append(질문)` → `inputs = final_state` (보따리 전체 전달) | `initial_state = {'messages': [질문]}` + `config` (방 번호만) |
| 2 프롬프트 · 노드 | `MessagesPlaceholder('chat_history')` · `state.messages[:-1]` (Day 053 부터 이미 있음) | 그대로 — DB 가 복원한 기록이 이 길로 LLM 에 들어간다 |
| 3 컴파일 | `graph.compile()` | `async with AsyncSqliteSaver.from_conn_string(db) as memory: graph.compile(checkpointer=memory)` |

In [9]:
docs = CSVLoader('faq_manual.csv', encoding='utf-8', content_columns=['question', 'answer'],
                 metadata_columns=['faq_id', 'category']).load()
print('문서 수 :', len(docs))

문서 수 : 10


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(model='gpt-5-nano')
model = OpenAIEmbeddings(model='text-embedding-3-small')

In [11]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask', encode_kwargs={'normalize_embeddings': True})

In [12]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 체크포인터가 무엇을 저장 · 복원하는지 확인용

GREETING = ('안녕', '고마', '감사', '반가')

def _fake(prompt_value):
    msgs = prompt_value.to_messages()
    system, last, history = msgs[0].content, msgs[-1].content, msgs[1:-1]
    if "'RAG' 또는 'NO_RAG'" in system:                       # 의도 분류
        return AIMessage('NO_RAG' if any(w in last for w in GREETING) else 'RAG')
    if '재구성' in system:                                     # 질문 재구성 : 앞 사용자 질문 + 지금 질문 (Day 053 규칙)
        prev = [m.content for m in history if isinstance(m, HumanMessage)]
        return AIMessage(f'{prev[-1]} {last}' if prev else last)
    if 'answer: ' in system:                                   # 매뉴얼 기반 답변
        first = system.split('answer: ')[-1].split('. ')[0].strip()
        return AIMessage(f'(모의 · 기록 {len(history)}개) {first}.')
    return AIMessage(f'(모의 일반 답변 · 기록 {len(history)}개) 네, 무엇이든 물어보세요!')

llm = RunnableLambda(_fake)

In [13]:
vectorstore = Chroma.from_documents(docs, model, collection_name='faq_ckpt')
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})

rephrase_prompt = ChatPromptTemplate.from_messages([
    ('system', '주어진 채팅 기록과 최신 질문이 있을 때, 맥락을 참조하는 모호한 질문을 기록 없이도 이해할 수 있는 명확한 질문으로 재구성하세요. 답하지 말고 질문만 반환하세요.'),
    MessagesPlaceholder('chat_history'), ('human', '{input}')])
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', "당신은 코드크래프터스 아카데미의 친절한 고객 지원 담당자입니다. 아래 [Context] 로 답하고, 없으면 '죄송하지만 매뉴얼에서 정보를 찾을 수 없습니다'라고 답하세요.\n\n[Context]\n{context}"),
    MessagesPlaceholder('chat_history'), ('human', '{input}')])         # 포인트 2 : 과거 기록이 들어갈 자리
rephrase_chain = rephrase_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm

class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []
    rephrased_query: str = ''
    context: str = ''

async def rephrase_node(state: AgentState) -> dict:
    history, question = state.messages[:-1], state.messages[-1].content
    query = await rephrase_chain.ainvoke({'chat_history': history, 'input': question}) if history else question
    print(f'  [에이전트 1] 기록 {len(history)}개 · 재구성 : {query}')
    return {'rephrased_query': query}

async def retriever_node(state: AgentState) -> dict:
    found = await retriever.ainvoke(state.rephrased_query)
    print(f"  [에이전트 2] 검색 : {[d.metadata['faq_id'] for d in found]}")
    return {'context': '\n\n'.join(d.page_content for d in found)}

async def answer_node(state: AgentState) -> dict:
    response = await qa_chain.ainvoke({'context': state.context, 'chat_history': state.messages[:-1],
                                       'input': state.messages[-1].content})
    print(f'  [에이전트 3] 답변 : {response.content[:60]}')
    return {'messages': [response]}

graph = StateGraph(AgentState)
graph.add_node('rephrase_node', rephrase_node); graph.add_node('retriever_node', retriever_node); graph.add_node('answer_node', answer_node)
graph.set_entry_point('rephrase_node')
graph.add_edge('rephrase_node', 'retriever_node'); graph.add_edge('retriever_node', 'answer_node'); graph.add_edge('answer_node', END);

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# 필기 main() — 직접 입력 ('exit' 로 종료)
db_file = 'multi_agent.sqlite'

async def main():
    async with AsyncSqliteSaver.from_conn_string(db_file) as memory:          # 포인트 3
        multi_agent = graph.compile(checkpointer=memory)
        config = {'configurable': {'thread_id': 'user-8888'}}
        print('[비동기 방식 멀티 에이전트] 가동 중... (종료: exit)')
        while True:
            user_input = input('사용자: ')
            if user_input.strip().lower() == 'exit':
                print('챗봇을 안전하게 종료합니다.'); break
            try:
                final_state = await multi_agent.ainvoke({'messages': [HumanMessage(content=user_input)]}, config=config)   # 포인트 1
            except Exception as e:
                print(f'오류가 발생했습니다: {e}'); continue
            print(f"AI 에이전트:\n{final_state['messages'][-1].content}\n" + '-' * 80)

await main()                 # .py 파일이면 asyncio.run(main())

In [15]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 체크포인터가 무엇을 저장 · 복원하는지 확인용
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

# 같은 main() 을 입력 목록으로 흉내 — 2번 실행해 '프로그램을 다시 켠' 상황도 확인
db_file = 'multi_agent_demo.sqlite'
if os.path.exists(db_file):
    os.remove(db_file)

async def main(questions, thread_id='user-8888'):
    async with AsyncSqliteSaver.from_conn_string(db_file) as memory:
        multi_agent = graph.compile(checkpointer=memory)
        config = {'configurable': {'thread_id': thread_id}}
        for user_input in questions:
            print(f'사용자: {user_input}')
            final_state = await multi_agent.ainvoke({'messages': [HumanMessage(content=user_input)]}, config=config)
            print(f"  → DB 에 쌓인 messages {len(final_state['messages'])}개\n")

print('===== 1번째 실행 =====')
await main(['수강 신청한 강의를 환불하고 싶어요', '그거 전액 다 받으려면 조건이 어떻게 되나요?'])
print('===== 프로그램을 다시 켜서 같은 thread_id 로 =====')
await main(['수강 신청한 강의를 환불하고 싶어요'])

===== 1번째 실행 =====
사용자: 수강 신청한 강의를 환불하고 싶어요
  [에이전트 1] 기록 0개 · 재구성 : 수강 신청한 강의를 환불하고 싶어요
  [에이전트 2] 검색 : ['FAQ-002']
  [에이전트 3] 답변 : (모의 · 기록 0개) 네, 환불 가능합니다.
  → DB 에 쌓인 messages 2개

사용자: 그거 전액 다 받으려면 조건이 어떻게 되나요?
  [에이전트 1] 기록 2개 · 재구성 : 수강 신청한 강의를 환불하고 싶어요 그거 전액 다 받으려면 조건이 어떻게 되나요?
  [에이전트 2] 검색 : ['FAQ-002']
  [에이전트 3] 답변 : (모의 · 기록 2개) 네, 환불 가능합니다.
  → DB 에 쌓인 messages 4개

===== 프로그램을 다시 켜서 같은 thread_id 로 =====
사용자: 수강 신청한 강의를 환불하고 싶어요
  [에이전트 1] 기록 4개 · 재구성 : 그거 전액 다 받으려면 조건이 어떻게 되나요? 수강 신청한 강의를 환불하고 싶어요


  [에이전트 2] 검색 : ['FAQ-002']
  [에이전트 3] 답변 : (모의 · 기록 4개) 네, 환불 가능합니다.
  → DB 에 쌓인 messages 6개



- 1번째 실행 : 2턴째 질문에 새 질문만 넣었는데 기록 2개가 복원돼 재구성 · 환불 FAQ(FAQ-002) — `inputs = final_state` 없이도 대화가 이어진다
- 📝 **다시 켰을 때** : 같은 파일 · 같은 `thread_id`(`user-8888` 고정)라 **새 대화의 첫 질문인데도 앞 기록 4개가 붙어** 재구성 LLM 이 불렸다(위 출력 '기록 4개'). 필기 `main()` 을 다시 실행할 때마다 지난 대화가 이어진다. 새 대화는 `thread_id` 를 바꾸거나(예 : `f'user-{uuid.uuid4()}'`) DB 파일을 지워야 한다
- 📝 **기록은 계속 쌓인다** — 턴마다 2개씩 늘어난 messages 가 매번 재구성 · 답변 프롬프트에 모두 들어간다. 대화가 길어지면 토큰 · 비용이 늘어나므로, 최근 N개만 넘기기(`trim_messages`) · 요약 같은 정리가 필요하다
- 체크포인트 DB(`*.sqlite`)에는 **대화 원문이 그대로** 저장된다 — 개인정보가 담길 수 있으니 저장소에 올리지 말 것

### 수업 실행 기록 (필기 `실습6`)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

db_file = "multi_agent.sqlite"
async def main():
    async with AsyncSqliteSaver.from_conn_string(db_file) as memory:
        multi_agent = graph.compile(checkpointer=memory)
        config = {"configurable": {"thread_id": "user-8888"}}
        print("[비동기 방식 멀티 에이전트] 가동 중... (종료: exit)")
        print("=" * 80)
        while True:
            try:
                user_input = input("사용자: ")
                if user_input.lower() == 'exit':
                    print("챗봇을 안전하게 종료합니다."); break
                initial_state = {"messages": [HumanMessage(content=user_input)]}
                final_state = await multi_agent.ainvoke(initial_state, config=config)
                print(f"AI 에이전트:\n{final_state['messages'][-1].content}")
                print("-" * 80)
            except Exception as e:
                print(f'오류가 발생했습니다.: {e}')
await main()

[비동기 방식 멀티 에이전트] 가동 중... (종료: exit)
에이전트 1: 질문 맥락 분석 및 재구성
--------------------------------------------------------------------------------
[결과] 재구성된 쿼리: 수강 신청한 강의를 환불하고 싶어요
에이전트 2: 관련 매뉴얼 검색
답변의 근거: 
question: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
answer: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.
--------------------------------------------------------------------------------
에이전트 3: 최종 답변 작성
--------------------------------------------------------------------------------
AI 에이전트:
네, 환불 가능합니다. 결제일로부터 7일 이내이며 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 환불을 진행하시려면 고객센터를 통해 문의해 주세요.

진행에 필요한 정보: 주문번호(또는 결제정보), 강의명, 결제일, 현재 수강률을 준비해 주시면 빠르게 처리해 드립니다. 원하시면 제가 필요한 정보를 정리해 드릴게요.
--------------------------------------------------------------------------------
에이전트 1: 질문 맥락 분석 및 재구성
--------------------------------------------------------------------------------
[결과] 재구성된 쿼리: 수강 신청한 강의를 전액 환불받으려면 어떤 조건이 필요하나요?
에이전트 2: 관련 매뉴얼 검색
답변의 근거: 
question: 수

- 1턴 : 기록이 없어 재구성 없이 그대로(`수강 신청한 강의를 환불하고 싶어요`) — 이 실행의 DB 는 비어 있었다
- 2턴 : 새 질문만 넣었지만 DB 에서 1턴이 복원돼 **'수강 신청한 강의를 전액 환불받으려면 어떤 조건이 필요하나요?'** 로 재구성 → 환불 FAQ → 조건 2가지(7일 이내 · 수강률 5% 미만) — FAQ 와 일치
- 📝 Day 054 와 달리 '고객센터로 전달해 드리겠습니다' 같은 약속은 없어졌지만 '원하시면 제가 필요한 정보를 정리해 드릴게요' 처럼 FAQ 밖의 제안은 여전히 한다

## 7. 부록 — 수업 벡터 저장소 `chroma_db_multi` 들여다보기

필기가 읽는 `chroma_db_multi` 폴더(같은 폴더의 [`chroma_db_multi/`](./chroma_db_multi/))는 SQLite 파일 하나 + UUID 이름 폴더(HNSW 색인) 로 되어 있다. **임베딩을 다시 만들지 않고** 저장된 OpenAI 벡터를 바로 읽을 수 있다.

In [17]:
import sqlite3, numpy as np
with sqlite3.connect('chroma_db_multi/chroma.sqlite3') as con:
    print('컬렉션 :', list(con.execute('select name, dimension from collections')))
    for seg_id, scope in con.execute('select id, scope from segments'):
        print(f'  {scope:<8} 세그먼트 {seg_id}', '← 같은 이름의 폴더에 벡터 색인' if scope == 'VECTOR' else '← chroma.sqlite3 안')
print('폴더 :', sorted(os.listdir('chroma_db_multi')))

class_db = Chroma(persist_directory='chroma_db_multi')            # 검색어를 임베딩하지 않을 것이라 embedding_function 없이
stored = class_db.get(include=['embeddings', 'metadatas'])
E = np.array(stored['embeddings']); rows = [m['row'] for m in stored['metadatas']]
print(f'문서 {len(rows)}개 · 벡터 {E.shape[1]}차원 · 길이 {np.linalg.norm(E, axis=1).min():.3f}~{np.linalg.norm(E, axis=1).max():.3f}')

refund = E[rows.index(1)]                                         # 환불 FAQ(row 1) 의 OpenAI 벡터로 검색
for doc, dist in class_db.similarity_search_by_vector_with_relevance_scores(refund.tolist(), k=3):
    print(f"  row {doc.metadata['row']}  거리 {dist:.3f}  {doc.page_content.splitlines()[0][10:40]}")

컬렉션 : [('langchain', 1536)]
  VECTOR   세그먼트 bedc952c-c555-4d6c-887e-1c5c80182057 ← 같은 이름의 폴더에 벡터 색인
  METADATA 세그먼트 4d909dbc-9bdd-4635-ac56-df11691d1e1c ← chroma.sqlite3 안
폴더 : ['bedc952c-c555-4d6c-887e-1c5c80182057', 'chroma.sqlite3']
문서 10개 · 벡터 1536차원 · 길이 1.000~1.000
  row 1  거리 0.000  수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻
  row 2  거리 0.899  강의를 다 들으면 수료증을 받을 수 있나요?
  row 6  거리 1.094  수강 기간에 제한이 있나요?


- `text-embedding-3-small` 벡터는 1536차원 · 길이 약 1 이다(Day 049). 질문을 새로 검색하려면 같은 OpenAI 모델로 질문을 임베딩해야 하므로 키가 필요하다 — 위에서는 저장된 벡터끼리만 비교했다
- 📝 수업 출력의 Document id(`67981998-…`)는 이 폴더의 id 와 다르다 — 이 폴더는 수업 뒤에 **다시 만든** 저장소다. 필기의 '폴더가 있으면 불러오기' 때문에 PC 마다 다른 저장소를 읽을 수 있다는 Day 053 의 주의와 같은 이야기
- 원래 폴더에는 0바이트 `link_lists.bin` 도 있었지만(문서가 10개뿐이라 상위 층이 비어 있음) GitHub 웹 업로드가 빈 파일을 받지 않아 빠져 있다. 위처럼 없어도 읽고 검색된다

## 정리

| 할 일 | 코드 |
|---|---|
| 설치 | `pip install langgraph-checkpoint-sqlite aiosqlite` |
| 연결 · 장착 | `async with AsyncSqliteSaver.from_conn_string('파일.sqlite') as memory: agent = graph.compile(checkpointer=memory)` |
| 실행 | `await agent.ainvoke({'messages': [HumanMessage(새 질문)]}, config={'configurable': {'thread_id': '방번호'}})` |
| 확인 | `await agent.aget_state(config)` · `agent.aget_state_history(config)` — 한 턴에 (입력 + 시작 + 노드 수)개 |
| 이어 가기 | 오류 뒤 `await agent.ainvoke(None, config)` |
| 주의 | config 없음 → `ValueError` · 블록 밖 → `no active connection` · 리듀서 없는 항목은 앞 턴 값 유지 · 같은 thread_id 는 다시 켜도 이어짐 · 기록은 계속 쌓임 |